In [1]:
!pip install torch

In [2]:
!pip install numpy


In [7]:
import torch.nn as nn

In [14]:
import torch

In [16]:
import numpy as np

In [8]:
INTENTS = ["open_app", "type_text", "open_and_type"]
intent_to_idx = {i: idx for idx, i in enumerate(INTENTS)}
idx_to_intent = {v: k for k, v in intent_to_idx.items()}

TRAIN_DATA = [
    ("открой блокнот", "open_app", [1, 0], 1),
    ("напиши привет", "type_text", [0, 1], 1),
    ("открой блокнот и напиши привет", "open_and_type", [1, 1], 1),
    ("блокнот привет", "open_and_type", [1, 1], 0),  # плохая формулировка
    ("что-то странное", "type_text", [0, 0], 0),
]

In [5]:
def tokenize(text):
    return text.lower().split()

def build_vocab(data):
    words = set()
    for text, _, _, _ in data:
        for w in tokenize(text):
            words.add(w)
    vocab = sorted(words)
    return vocab, {w: i for i, w in enumerate(vocab)}

VOCAB, WORD_TO_IDX = build_vocab(TRAIN_DATA)
VOCAB_SIZE = len(VOCAB)

def text_to_vec(text):
    v = np.zeros(VOCAB_SIZE, dtype=np.float32)
    for w in tokenize(text):
        if w in WORD_TO_IDX:
            v[WORD_TO_IDX[w]] = 1.0
    return v

In [9]:
class IntentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(VOCAB_SIZE, 32),
            nn.ReLU(),
            nn.Linear(32, len(INTENTS))
        )

    def forward(self, x):
        return self.net(x)

In [10]:
class EntityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(VOCAB_SIZE, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class ConfidenceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [17]:
X = torch.tensor([text_to_vec(t) for t, _, _, _ in TRAIN_DATA])
y_intent = torch.tensor([intent_to_idx[i] for _, i, _, _ in TRAIN_DATA])
y_entity = torch.tensor([e for _, _, e, _ in TRAIN_DATA], dtype=torch.float32)
y_conf = torch.tensor([[c] for _, _, _, c in TRAIN_DATA], dtype=torch.float32)

intent_model = IntentNet()
entity_model = EntityNet()
conf_model = ConfidenceNet()

opt_i = torch.optim.Adam(intent_model.parameters(), lr=0.01)
opt_e = torch.optim.Adam(entity_model.parameters(), lr=0.01)
opt_c = torch.optim.Adam(conf_model.parameters(), lr=0.01)

loss_i = nn.CrossEntropyLoss()
loss_e = nn.BCELoss()
loss_c = nn.BCELoss()

/tmp/ipython-input-1546313970.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X = torch.tensor([text_to_vec(t) for t, _, _, _ in TRAIN_DATA])


In [19]:
for epoch in range(400):
    opt_i.zero_grad()
    out_i = intent_model(X)
    li = loss_i(out_i, y_intent)
    li.backward()
    opt_i.step()

    opt_e.zero_grad()
    out_e = entity_model(X)
    le = loss_e(out_e, y_entity)
    le.backward()
    opt_e.step()

    opt_c.zero_grad()
    probs = torch.softmax(out_i.detach(), dim=1)
    conf_input = torch.cat([probs, out_e.detach()], dim=1)
    out_c = conf_model(conf_input)
    lc = loss_c(out_c, y_conf)
    lc.backward()
    opt_c.step()

print("Обучение закончено")

Обучение закончено


In [20]:
def run_test(text):
    x = torch.tensor(text_to_vec(text)).unsqueeze(0)

    with torch.no_grad():
        logits = intent_model(x)
        probs = torch.softmax(logits, dim=1)[0]
        intent = idx_to_intent[torch.argmax(probs).item()]

        entities = entity_model(x)[0]
        has_app = entities[0].item()
        has_text = entities[1].item()

        conf_input = torch.cat([probs, entities])
        confidence = conf_model(conf_input.unsqueeze(0))[0][0].item()

    print("\nКоманда:", text)
    print("IntentNet = ", intent)
    print(f"EntityNet app={has_app:.2f}, text = {has_text:.2f}")
    print(f"ConfidenceNet уверенность = {confidence:.2f}")

    if confidence < 0.5:
        print("уточните команду.")
    else:
        print("команда верна")

In [21]:
run_test("Открой пожалуйста блокнот")


Команда: Открой пожалуйста блокнот
IntentNet =  open_app
EntityNet app=1.00, text = 0.00
ConfidenceNet уверенность = 1.00
команда верна


In [24]:
run_test("Привет! калькулятор открой")


Команда: Привет! калькулятор открой
IntentNet =  open_app
EntityNet app=0.97, text = 0.00
ConfidenceNet уверенность = 1.00
команда верна


In [25]:
run_test("Можешь пожалуйста написать Привет мир")


Команда: Можешь пожалуйста написать Привет мир
IntentNet =  type_text
EntityNet app=0.59, text = 1.00
ConfidenceNet уверенность = 1.00
команда верна


In [27]:
run_test("непонятная команда какая-то")


Команда: непонятная команда какая-то
IntentNet =  type_text
EntityNet app=0.40, text = 0.24
ConfidenceNet уверенность = 0.09
уточните команду.
